In [1]:
!pip install gradio sentence-transformers numpy -q

import gradio as gr
import numpy as np
from sentence_transformers import SentenceTransformer

# ---------- Load AI Model ----------
model = SentenceTransformer('all-MiniLM-L6-v2')


# ---------- Similarity Function ----------
def similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


# ---------- Skill Analysis ----------
def analyze(resume, job):

    stop_words = {
        "and","or","the","a","an","with","in","on","for",
        "to","is","are","we","you","i","of"
    }

    r_words = resume.lower().replace(",", "").replace(".", "").split()
    j_words = job.lower().replace(",", "").replace(".", "").split()

    r_clean = [w for w in r_words if w not in stop_words]
    j_clean = [w for w in j_words if w not in stop_words]

    match = []
    missing = []

    for w in j_clean:
        if w in r_clean:
            match.append(w)
        else:
            missing.append(w)

    return list(set(match))[:10], list(set(missing))[:10]


# ---------- Dashboard ----------
def dashboard(resume, job):

    # Convert text into embeddings
    r_vec = model.encode(resume)
    j_vec = model.encode(job)

    # Calculate similarity
    score = similarity(r_vec, j_vec) * 100

    # Find matching & missing skills
    match, missing = analyze(resume, job)

    # Dashboard UI
    return f"""
    <div style="
        font-family:Arial;
        padding:20px;
        background:#f8fafc;
        color:#111827;
        border-radius:15px;
        border:2px solid #e5e7eb;
    ">

        <h1 style="
            color:#2563eb;
            text-align:center;
        ">
        📊 AI Semantic Skill Matching and Gap Analyzer
        </h1>

        <p style="
            text-align:center;
            color:#6b7280;
            font-size:16px;
        ">
        AI-powered Resume vs Job Description Matching System
        </p>

        <div style="
            background:#e0f2fe;
            padding:15px;
            border-radius:10px;
            margin-top:15px;
            border-left:6px solid #38bdf8;
        ">
            <h2 style="color:#0284c7;">📈 Match Score</h2>
            <h1 style="color:#16a34a;">{round(score,2)}%</h1>
        </div>

        <div style="
            display:flex;
            gap:15px;
            margin-top:15px;
        ">

            <div style="
                flex:1;
                background:#ecfdf5;
                padding:15px;
                border-radius:10px;
                border:1px solid #a7f3d0;
            ">
                <h3 style="color:#16a34a;">🧠 Matching Skills</h3>
                <p>{', '.join(match) if match else 'None'}</p>
            </div>

            <div style="
                flex:1;
                background:#fef2f2;
                padding:15px;
                border-radius:10px;
                border:1px solid #fecaca;
            ">
                <h3 style="color:#dc2626;">🔍 Missing Skills</h3>
                <p>{', '.join(missing) if missing else 'None'}</p>
            </div>

        </div>

        <div style="
            margin-top:15px;
            background:#f1f5f9;
            padding:10px;
            border-radius:10px;
            text-align:center;
            color:#475569;
        ">
        💡 This system uses AI embeddings for semantic similarity comparison.
        </div>

    </div>
    """


# ---------- Gradio Interface ----------
app = gr.Interface(
    fn=dashboard,

    inputs=[
        gr.Textbox(
            lines=8,
            label="📄 Resume",
            placeholder="Paste resume text here..."
        ),

        gr.Textbox(
            lines=8,
            label="📄 Job Description",
            placeholder="Paste job description here..."
        )
    ],

    outputs=gr.HTML(),

    title="📊 AI Semantic Skill Matching and Gap Analyzer",

    description="Compare Resume and Job Description using AI Embeddings"
)


# ---------- Launch App ----------
app.launch(share=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3fa7f87cf5e23e6a56.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
